In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

file = "../../data/HCMST 2017 to 2022 small public version 2.2.dta"
df = pd.read_stata(file)

/var/folders/vp/lwy9wn811sj3bwmkzg6q0rgm0000gn/T/ipykernel_11608/3661410484.py:11: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df = pd.read_stata(file)


In [7]:
# 1. Filter for 'Survivors' (Those who completed Wave 1 and Wave 3)
# We look for people with a valid entry in w3_section (the status variable)
long_df = df[df['w3_section'].notna()].copy()
long_df['w1_identity_all_modified'].value_counts()  # Check the distribution of the identity variable

w1_identity_all_modified
heterosexual or straight    1457
bisexual                     116
gay                          112
lesbian                       25
Something else                11
refused or unknown             1
Name: count, dtype: int64

In [8]:
# 2. Define the Orientation Groups
# Using the 'w1_identity_all_modified' variable
long_df['group'] = long_df['w1_identity_all_modified'].apply(
    lambda x: 'LGB' if any(i in str(x).lower() for i in ['gay', 'lesbian', 'bisexual', 'Something else', 'refused or unknown']) else 'Straight'
)

# 3. Standardize Quality Scores (Converting categories to 1-5 scale)
# Excellent=5, Very Good=4, Good=3, Fair=2, Poor=1
quality_map = {'Excellent': 5, 'Very Good': 4, 'Good': 3, 'Fair': 2, 'Poor': 1}

long_df['q_w1'] = long_df['w1_q34'].map(quality_map)
long_df['q_w2'] = long_df['w2_rel_qual_combo'].map(quality_map)
long_df['q_w3'] = long_df['w3_rel_qual'].map(quality_map)

# 4. Calculate Background 'Match Density'
# How many background points did they match on originally?
long_df['match_score'] = long_df[['pol_match', 'edu_match', 'race_match']].sum(axis=1)

KeyError: "None of [Index(['pol_match', 'edu_match', 'race_match'], dtype='str')] are in the [columns]"